In [0]:
%pip install databricks_langchain==0.9.0 -q
%pip install databricks-vectorsearch -q
%pip install mlflow-skinny[databricks] -q
%pip install langchain==1.0.0 -q
%pip install langgraph==1.0.10 -q

dbutils.library.restartPython()

In [0]:
import json
import mlflow
import mlflow.deployments
import pyspark.sql.functions as sf
import pyspark.sql.types as st

from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from databricks.vector_search.client import VectorSearchClient
from databricks.vector_search.reranker import DatabricksReranker
from langchain.agents import create_agent

In [0]:
df = (
  spark
  .read
  .table("workspace.sephora_products_and_skincare_reviews.bronze_products_info")
)
display(df.limit(5))

In [0]:
df_recommend_bronze = (
  df
  .select("product_id", "product_name", "primary_category", "secondary_category", "tertiary_category", "highlights")
  .withColumn(
    "product_summary",
    sf.to_json(
      sf.struct(
        "primary_category", "secondary_category", "tertiary_category", "highlights"
      )
    )
  )
)
display(df_recommend_bronze.limit(5))

In [0]:
df_recommend_bronze.write.mode("overwrite").saveAsTable("workspace.sephora_products_and_skincare_reviews.bronze_products_recommendations")

In [0]:
@sf.udf(st.StringType())
def process_json(json_str: str) -> str:
    try:
        obj = json.loads(json_str)
        if not isinstance(obj, dict):
            raise ValueError("JSON is not a dictionary")
        obj_str = ""
        for key, value in obj.items():
            obj_str += f"{key}:\n\t{value}\n"
        return obj_str
    except Exception:
        return json_str

In [0]:
df_recommend_silver = df_recommend_bronze.withColumn("product_summary", process_json("product_summary"))

In [0]:
df_recommend_silver.limit(30).write.mode("overwrite").saveAsTable("workspace.sephora_products_and_skincare_reviews.silver_products_recommendations")

In [0]:
%sql
USE workspace.sephora_products_and_skincare_reviews;

ALTER TABLE silver_products_recommendations
  SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

SELECT * FROM silver_products_recommendations;

In [0]:
deploy_client = mlflow.deployments.get_deploy_client("databricks")

question = "What are product's features?"
response = deploy_client.predict(
    endpoint="databricks-gte-large-en",
    inputs={"input": [question]}
)

embeddings = [
    e["embedding"]
    for e
    in  response.data
]

In [0]:
print("Embedding shape:", len(embeddings[0]))
print("Embeddings for question:", embeddings[0])

In [0]:
vsc = VectorSearchClient()

vector_search_endpoint = "sephora"
index_name = "workspace.sephora_products_and_skincare_reviews.silver_products_recommendations_index"
table_name = "workspace.sephora_products_and_skincare_reviews.silver_products_recommendations"

In [0]:
vsc.create_delta_sync_index_and_wait(
    endpoint_name=vector_search_endpoint,
    index_name=index_name,
    source_table_name=table_name,
    primary_key="product_id",
    embedding_source_column="product_summary",
    embedding_model_endpoint_name="databricks-gte-large-en",
    pipeline_type="TRIGGERED",
)

In [0]:
index = vsc.get_index(index_name=index_name)
print(
    json.dumps(
        index.describe(), 
        indent=4
))

In [0]:
query_text = "Some fragrance with woody, earthy scent"
results = index.similarity_search(
    query_text=query_text,
    columns=["product_id", "product_name", "primary_category", "secondary_category", "tertiary_category", "highlights"],
    num_results=10,
)

display(results)

In [0]:
query_text = "Some fragrance with woody, earthy scent"
results = index.similarity_search(
    query_text=query_text,
    query_type="hybrid",
    columns=["product_id", "product_name", "primary_category", "secondary_category", "tertiary_category", "highlights"],
    num_results=10,
)

display(results)

In [0]:
query_text = "Some fragrance with woody, earthy scent"
results = index.similarity_search(
    query_text=query_text,
    query_type="FULL_TEXT",
    columns=["product_id", "product_name", "primary_category", "secondary_category", "tertiary_category", "highlights"],
    num_results=10,
)

display(results)

In [0]:
query_text = "woody, earthy"
results = index.similarity_search(
    query_text=query_text,
    query_type="FULL_TEXT",
    filters={
        "secondary_category": "Value & Gift Sets"
    },
    columns=["product_id", "product_name", "primary_category", "secondary_category", "tertiary_category", "highlights"],
    num_results=10,
)

display(results)

In [0]:
query_text = "Some fragrance with woody, earthy scent"
results = index.similarity_search(
    query_text=query_text,
    query_type="FULL_TEXT",
    columns=["product_id", "product_name", "primary_category", "secondary_category", "tertiary_category", "highlights"],
    num_results=10,
    reranker=DatabricksReranker(columns_to_rerank=["product_summary"]),
)

display(results)

In [0]:
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.state import CompiledStateGraph

mlflow.langchain.autolog()
LLM_ENDPOINT_NAME = "databricks-gpt-oss-120b"

In [0]:
llm_endpoint_name = "databricks-gpt-oss-120b"

def build_agent(llm_endpoint: str, index_name: str, num_results: int = 3) -> CompiledStateGraph:
    model = ChatDatabricks(
        endpoint=llm_endpoint_name,
        max_tokens=500
    )

    vs_tool = VectorSearchRetrieverTool(
        index_name=index_name,
        description="A vector store of sephora products",
        num_results=num_results,
    )

    system_prompt = """You are expert on Sephora's products. Respond in a clear, professional, and factual tone
    appropriate for engineers and technical staff. Use only verified information from the provided context, include 
    reference to every information you provide. Do not hallucinate information. If you are unsure about the answer,
    say "I don't know"."""
    checkpointer = InMemorySaver()

    agent = create_agent(
        model,
        tools=[vs_tool],
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )

    return agent

config = {"configurable": {"thread_id": "sephora-demo"}}

agent = build_agent(llm_endpoint=LLM_ENDPOINT_NAME, index_name=index_name, num_results=3)

In [0]:
config = {"configurable": {"thread_id": "sephora_products_test"}}
response = agent.invoke(
    input={
        "messages": [
            {
                "role": "user",
                "content": "What products are planet positive?"
            }
        ]
    },
    config=config
)
print(response["messages"][-1].content)

| Product ID | Primary Category | Secondary Category | Tertiary Category | Highlights (excerpt) |
|------------|------------------|--------------------|-------------------|----------------------|
| **P483139** | Fragrance | Women | Perfume | “Clean + Planet Positive”, Vegan, Cruelty‑Free, Fresh Scent, Without Parabens, Without Phthalates【0†page_content】 |
| **P483079** | Fragrance | Women | Perfume | “Clean + Planet Positive”, Vegan, Fresh Scent, Without Phthalates, Without Parabens, Cruelty‑Free【0†page_content】 |
| **P483109** | Fragrance | Women | Perfume | “Clean + Planet Positive”, Vegan, Cruelty‑Free, Woody & Earthy Scent, Without Parabens